In [3]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# El parámetro read_only=True permite abrirla aunque otro proceso la esté usando
con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

In [ ]:
import pandas as pd

# 1. Obtener la lista de tablas del esquema principal
tablas = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").df()['table_name'].tolist()

print("="*70)
print(f"CLASIFICACIÓN ESTRUCTURAL DEL DATASET")
print("="*70)

for tabla in tablas:
    # Traemos el esquema de la tabla (solo 1 fila para no gastar memoria)
    df_temp = con.execute(f"SELECT * FROM main.{tabla} LIMIT 1").df()
    
    # Clasificación por tipos de Pandas
    v_texto = df_temp.select_dtypes(include=['object', 'category']).columns.tolist()
    v_num = df_temp.select_dtypes(include=['number']).columns.tolist()
    v_fecha = df_temp.select_dtypes(include=['datetime', 'datetime64', 'datetimetz']).columns.tolist()

    print(f"\nTABLA: {tabla.upper()}")
    
    if v_texto:
        print(f"   🔤 TEXTO/CATEGORÍA: {', '.join(v_texto)}")
    
    if v_num:
        print(f"   🔢 NUMÉRICAS:       {', '.join(v_num)}")
        
    if v_fecha:
        print(f"   📅 FECHAS:          {', '.join(v_fecha)}")
    
    print("-" * 40)

print("\n" + "="*70)
print("Clasificación finalizada.")

CLASIFICACIÓN ESTRUCTURAL DEL DATASET

TABLA: CAUSA
   🔤 TEXTO/CATEGORÍA: causa, causa_especifica
   🔢 NUMÉRICAS:       id_causa
----------------------------------------

TABLA: CLIMATOLOGIA
   🔤 TEXTO/CATEGORÍA: id_variable, id_clave_inc
   🔢 NUMÉRICAS:       id_registro, resultado_numerico
   📅 FECHAS:          fecha_de_observacion
----------------------------------------

TABLA: DANOS
   🔤 TEXTO/CATEGORÍA: id_clave_inc, tamanio
   🔢 NUMÉRICAS:       hojarasca, arbustivo, herbaceo, arbolado_adulto, renuevo
----------------------------------------

TABLA: DEMOGRAFIA
   🔤 TEXTO/CATEGORÍA: id_cvegeo, sexo
   🔢 NUMÉRICAS:       anio, pob_total, r_00_04, r_05_09, r_10_14, r_15_19, r_20_24, r_25_29, r_30_34, r_35_39, r_40_44, r_45_49, r_50_54, r_55_59, r_60_64, r_65_69, r_70_74, r_75_79, r_80_84, r_85_mm
----------------------------------------

TABLA: DICCIONARIO
   🔤 TEXTO/CATEGORÍA: id_variable, fuente, nombre_completo, unidad_de_medida
----------------------------------------

TABLA: E

In [ ]:
import pandas as pd

tablas = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").df()['table_name'].tolist()

print("="*80)
print(f"🕵️  INSPECCIÓN DE CAMPOS DE TEXTO PARA ESTRATEGIA DE LIMPIEZA")
print("="*80)

for tabla in tablas:
    # Obtenemos columnas de tipo texto (VARCHAR/STRING)
    cols_texto = con.execute(f"""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = '{tabla}' AND data_type = 'VARCHAR'
    """).df()['column_name'].tolist()
    
    if cols_texto:
        print(f"\n📂 TABLA: {tabla.upper()}")
        # Tomamos una muestra para ver cómo lucen los datos actuales
        muestra = con.execute(f"SELECT {', '.join([f'\"{c}\"' for c in cols_texto])} FROM main.{tabla} LIMIT 5").df()
        display(muestra)
        print("-" * 80)

🕵️  INSPECCIÓN DE CAMPOS DE TEXTO PARA ESTRATEGIA DE LIMPIEZA

📂 TABLA: CAUSA


,causa,causa_especifica
0,intencional,cambio de uso de suelo
1,quema de basureros,basurero irregular
2,transportes,escapes
3,otras actividades productivas,lineas electricas
4,fumadores,fumadores


--------------------------------------------------------------------------------

📂 TABLA: CLIMATOLOGIA


,id_variable,id_clave_inc
0,T2M,15-15-0002
1,T2M,15-15-0002
2,T2M,15-15-0002
3,T2M,15-15-0002
4,T2M,15-15-0002


--------------------------------------------------------------------------------

📂 TABLA: DANOS


...

--------------------------------------------------------------------------------

📂 TABLA: OPERACIONES


,id_clave_inc
0,15-15-0010
1,15-15-0020
2,15-15-0021
3,15-15-0028
4,15-15-0030


--------------------------------------------------------------------------------

📂 TABLA: VEGETACION


,regimen_del_fuego,tipo_de_vegetacion
0,otros,mezquital espinoso
1,otros,matorral crasicaule
2,adaptado,matorral espinoso tamaulipeco
3,adaptado,pradera de alta montana
4,otros,matorral sarco-crasicaule


--------------------------------------------------------------------------------


In [ ]:
# Diagnóstico de Salud de los Datos (Sin borrar nada)
print("="*60)
print("🔎 REPORTE DE INCONSISTENCIAS LÓGICAS")
print("="*60)

# 1. Error de Tiempos
tiempos_err = con.execute("""
    SELECT COUNT(*) FROM main.operaciones o
    JOIN main.incendios i ON o.id_clave_inc = i.id_clave_inc
    WHERE o.fecha_termino < i.fecha_inicio 
       OR (o.fecha_termino = i.fecha_inicio AND o.hora_llegada < o.hora_deteccion)
""").fetchone()[0]

# 2. Error de Coordenadas
gps_err = con.execute("""
    SELECT COUNT(*) FROM main.incendios 
    WHERE latitud NOT BETWEEN 14 AND 33 OR longitud NOT BETWEEN -118 AND -86
""").fetchone()[0]

# 3. Error de Hectáreas
danos_err = con.execute("""
    SELECT COUNT(*) FROM main.danos 
    WHERE hojarasca < 0 OR arbustivo < 0 OR herbaceo < 0 OR arbolado_adulto < 0 OR renuevo < 0
""").fetchone()[0]

# 4. Error de Demografía
demo_err = con.execute("""
    SELECT COUNT(*) FROM main.demografia 
    WHERE pob_total <= 0
""").fetchone()[0]

print(f"⏰ Tiempos Imposibles:    {tiempos_err} registros")
print(f"📍 GPS fuera de rango:   {gps_err} registros")
print(f"🔥 Daños Negativos:      {danos_err} registros")
print(f"👥 Población Inválida:   {demo_err} registros")
print("="*60)

🔎 REPORTE DE INCONSISTENCIAS LÓGICAS
⏰ Tiempos Imposibles:    114 registros
📍 GPS fuera de rango:   0 registros
🔥 Daños Negativos:      0 registros
👥 Población Inválida:   0 registros


In [ ]:
import pandas as pd

def inspeccionar_salud_estructural(con):
    print("="*85)
    print(f"🔍 DIAGNÓSTICO ESTRATÉGICO (IDs PROTEGIDOS)")
    print("="*85)

    # 1. Lista de nombres de columnas que funcionan como llaves (IDs)
    # Agregamos todas las que definiste en tu SQL
    ids_sagrados = [
        'id_clave_inc', 'id_cvegeo', 'id_variable', 'id_causa', 
        'id_vegetacion', 'id_clave_ent', 'id_registro'
    ]

    tablas_df = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").df()
    
    if tablas_df.empty:
        print("❌ No se encontraron tablas en 'main'.")
        return

    for tabla in tablas_df['table_name'].tolist():
        total_filas = con.execute(f"SELECT COUNT(*) FROM main.{tabla}").fetchone()[0]
        
        columnas_info = con.execute(f"""
            SELECT column_name, data_type 
            FROM information_schema.columns 
            WHERE table_name = '{tabla}'
        """).df()

        resultados = []

        for _, row in columnas_info.iterrows():
            col = row['column_name']
            tipo = row['data_type']
            tipo_up = tipo.upper()
            
            nulos = con.execute(f'SELECT COUNT(*) FROM main.{tabla} WHERE "{col}" IS NULL').fetchone()[0]
            porcentaje = round((nulos / total_filas) * 100, 2) if total_filas > 0 else 0
            
            # --- LÓGICA DE CLASIFICACIÓN REFINADA ---
            
            # 1. Prioridad: ¿Es una llave/ID?
            if col.lower() in ids_sagrados:
                accion = "🔒 ID Protegido (No tocar)"
            
            # 2. Si es texto descriptivo (y no es ID)
            elif 'VARCHAR' in tipo_up or 'STRING' in tipo_up:
                accion = "🏷️ Etiquetar ('no especificado')"
            
            # 3. Si es métrica numérica
            elif any(x in tipo_up for x in ['INT', 'FLOAT', 'DOUBLE', 'DECIMAL']):
                accion = "🔢 Preservar NULL (Métrica)"
            
            # 4. Si es tiempo o fecha
            elif any(x in tipo_up for x in ['DATE', 'TIME', 'TIMESTAMP', 'INTERVAL']):
                accion = "📅 Preservar NULL (Temporal)"
            else:
                accion = "❓ Revisar"

            resultados.append({
                'Columna': col,
                'Tipo SQL': tipo,
                'Nulos': nulos,
                '% Nulos': porcentaje,
                'Acción sugerida': accion
            })

        print(f"\n📂 TABLA: {tabla.upper()} | Registros: {total_filas}")
        print("-" * 85)
        df_reporte = pd.DataFrame(resultados)
        print(df_reporte.to_string(index=False))

# EJECUCIÓN
inspeccionar_salud_estructural(con)

🔍 DIAGNÓSTICO ESTRATÉGICO (IDs PROTEGIDOS)



📂 TABLA: CAUSA | Registros: 58
-------------------------------------------------------------------------------------
         Columna Tipo SQL  Nulos  % Nulos                  Acción sugerida
        id_causa  INTEGER      0      0.0        🔒 ID Protegido (No tocar)
           causa  VARCHAR      0      0.0 🏷️ Etiquetar ('no especificado')
causa_especifica  VARCHAR      0      0.0 🏷️ Etiquetar ('no especificado')

📂 TABLA: CLIMATOLOGIA | Registros: 14132040
-------------------------------------------------------------------------------------
             Columna Tipo SQL  Nulos  % Nulos             Acción sugerida
         id_registro   BIGINT      0      0.0   🔒 ID Protegido (No tocar)
         id_variable  VARCHAR      0      0.0   🔒 ID Protegido (No tocar)
        id_clave_inc  VARCHAR      0      0.0   🔒 ID Protegido (No tocar)
fecha_de_observacion     DATE      0      0.0 📅 Preservar NULL (Temporal)
  resultado_numerico    FLOAT      0      0.0  🔢 Preservar NULL (Métrica)

📂 TABL

In [ ]:
import pandas as pd

def inspeccionar_climatologia_profunda(con):
    print("="*90)
    print("ANALISIS ESTRATEGICO DE LA TABLA CLIMATOLOGIA (11M+ REGISTROS)")
    print("="*90)

    # 1. Conteo general y salud de la tabla
    res_general = con.execute("""
        SELECT 
            COUNT(*) as total_registros,
            COUNT(DISTINCT id_variable) as total_variables,
            COUNT(*) FILTER (WHERE resultado_numerico IS NULL) as nulos_totales
        FROM main.climatologia
    """).df()
    
    print("\n--- RESUMEN GENERAL ---")
    print(res_general.to_string(index=False))

    # 2. Estadisticos descriptivos por variable
    # Esto es vital para definir los umbrales de neutralizacion
    print("\n--- DISTRIBUCION POR VARIABLE (LIMITES FISICOS) ---")
    stats_query = """
        SELECT 
            id_variable,
            COUNT(*) as registros,
            MIN(resultado_numerico) as min_valor,
            MAX(resultado_numerico) as max_valor,
            AVG(resultado_numerico) as promedio,
            STDDEV(resultado_numerico) as desv_std,
            COUNT(*) FILTER (WHERE resultado_numerico IS NULL) as nulos
        FROM main.climatologia
        GROUP BY id_variable
        ORDER BY registros DESC
    """
    df_stats = con.execute(stats_query).df()
    
    # Calculamos el porcentaje de nulos por variable
    df_stats['%_nulos'] = (df_stats['nulos'] / df_stats['registros'] * 100).round(2)
    
    print(df_stats.to_string(index=False))

    # 3. Deteccion de posibles valores "basura" (Outliers extremos)
    # Buscamos valores que se alejan mas de 4 desviaciones estandar
    print("\n--- ANALISIS DE VALORES EXTREMOS (POSIBLES ERRORES DE SENSOR) ---")
    for var in df_stats['id_variable'].unique():
        sub_stats = df_stats[df_stats['id_variable'] == var].iloc[0]
        umbral_superior = sub_stats['promedio'] + (4 * sub_stats['desv_std'])
        umbral_inferior = sub_stats['promedio'] - (4 * sub_stats['desv_std'])
        
        print(f"\nVariable: {var}")
        print(f"   Umbral sugerido (+/- 4 sigma): [{round(umbral_inferior, 2)} a {round(umbral_superior, 2)}]")
        
# Ejecucion de la inspeccion
inspeccionar_climatologia_profunda(con)

ANALISIS ESTRATEGICO DE LA TABLA CLIMATOLOGIA (11M+ REGISTROS)

--- RESUMEN GENERAL ---
 total_registros  total_variables  nulos_totales
        14132040              145              0

--- DISTRIBUCION POR VARIABLE (LIMITES FISICOS) ---
                id_variable  registros   min_valor   max_valor    promedio   desv_std  nulos  %_nulos
                PRECTOTCORR     197130    0.000000   46.049999    0.483035   1.285788      0      0.0
                 TOA_SW_DNI      98570   54.720001   66.500000   61.646128   1.894536      0      0.0
                  WS50M_MAX      98570    1.580000   19.770000    5.605787   1.543802      0      0.0
                 GWM_HEIGHT      98570 -999.000000 -999.000000 -999.000000   0.000000      0      0.0
               MIDDAY_INSOL      98570    5.270000   98.650002   81.448364  10.675078      0      0.0
                      SNODP      98570 -999.000000   10.650000  -10.625686 102.558662      0      0.0
                        SZA      98570 -999.000

In [ ]:
def investigar_diccionario_clima(con):
    print("\n--- DEFINICIONES Y UNIDADES DEL DICCIONARIO ---")
    query = """
        SELECT id_variable, nombre_completo, unidad_de_medida, fuente
        FROM main.diccionario
        WHERE id_variable IN (
            SELECT DISTINCT id_variable FROM main.climatologia
        )
        ORDER BY id_variable
    """
    df_dicc = con.execute(query).df()
    # Mostramos todo para analizarlo juntos
    import pandas as pd
    pd.set_option('display.max_colwidth', None)
    print(df_dicc.to_string(index=False))

investigar_diccionario_clima(con)


--- DEFINICIONES Y UNIDADES DEL DICCIONARIO ---
                id_variable                                               nombre_completo      unidad_de_medida              fuente
                    AIRMASS                                                      air mass         dimensionless                nasa
                  ALLSKY_KT                            all sky insolation clearness index         dimensionless                nasa
                 ALLSKY_NKT                 all sky normalized insolation clearness index         dimensionless                nasa
          ALLSKY_SFC_LW_DWN                  all sky surface longwave downward irradiance            mj/m^2/day                nasa
           ALLSKY_SFC_LW_UP                    all sky surface longwave upward irradiance            mj/m^2/day                nasa
        ALLSKY_SFC_PAR_DIFF                                   all sky surface diffuse par            mj/m^2/day                nasa
        ALLSKY_SFC_PAR_DIRH

# Inspección para la estandarización de datos tipo texto 

- Eliminar acentos extraños en los textos.
- Convertir el formato a mayúsculas.
- Inspeccionar si existen valores con espacios invisibles.

In [ ]:
## ESTA CONSULTA BUSCARA EN TODA LA BASE DE DATOS LAS VARIABLES QUE VIENEN EN FORMATO DE TEXTO 
text_columns = con.execute("""
    SELECT table_name, column_name 
    FROM information_schema.columns 
    WHERE data_type = 'VARCHAR' 
      AND table_schema = 'main'
""").fetchall()

print(f"Auditando {len(text_columns)} columnas de texto...\n")

for table, col in text_columns:
    stats = con.execute(f"""
        SELECT 
            COUNT(*) as total,
            SUM(CASE WHEN "{col}" != TRIM("{col}") THEN 1 ELSE 0 END) as con_espacios,
            SUM(CASE WHEN regexp_matches("{col}", '[áéíóúÁÉÍÓÚüÜñÑ]') THEN 1 ELSE 0 END) as con_acentos_especiales,
            SUM(CASE WHEN "{col}" != UPPER("{col}") AND "{col}" != LOWER("{col}") THEN 1 ELSE 0 END) as mezcla_case
        FROM {table}
    """).fetchone()
    
    if stats[1] > 0 or stats[2] > 0 or stats[3] > 0:
        print(f"Tabla: {table} | Columna: {col}")
        print(f"   - Espacios extra (inicio/fin): {stats[1]}")
        print(f"   - Con acentos/caracteres: {stats[2]}")
        print(f"   - Mezcla Mayúsculas/Minúsculas: {stats[3]}")
        print("-" * 30)

Auditando 22 columnas de texto...



#### Codificar una funcion en transform para limpiar todo tipo de caracteres raros y estandarizarlos para que el modelo no se confunda 

# Toma de Decisiones

In [ ]:
#Buscar las relaciones entre las tablas

con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

# 1. Consultamos el catálogo de llaves foráneas
# DuckDB guarda las relaciones en la tabla 'duckdb_constraints'
query_relaciones = """
    SELECT 
        table_name AS tabla_hija,
        constraint_column_names[1] AS columna_fk,
        referenced_table AS tabla_padre
    FROM duckdb_constraints
    WHERE constraint_type = 'FOREIGN KEY'
"""

relaciones_detectadas = con.execute(query_relaciones).fetchall()

# 2. Convertimos el resultado al formato que necesitamos para la auditoría
relaciones = []
print("🔍 RELACIONES DETECTADAS EN EL ESQUEMA:")
print(f"{'TABLA HIJA':<15} | {'COLUMNA FK':<15} | {'TABLA PADRE':<15}")
print("-" * 50)

for hija, fk, padre in relaciones_detectadas:
    # Como tus PK suelen llamarse igual que las FK, las mapeamos así:
    # Nota: Si una PK se llama diferente, DuckDB también nos lo puede decir
    relaciones.append((hija, fk, padre, fk)) 
    print(f"{hija:<15} | {fk:<15} | {padre:<15}")

# Ahora 'relaciones' ya no es una lista manual, ¡es dinámica!

🔍 RELACIONES DETECTADAS EN EL ESQUEMA:
TABLA HIJA      | COLUMNA FK      | TABLA PADRE    
--------------------------------------------------
climatologia    | id_variable     | diccionario    
climatologia    | id_clave_inc    | incendios      
danos           | id_clave_inc    | incendios      
demografia      | id_cvegeo       | municipios     
incendios       | id_cvegeo       | municipios     
incendios       | id_causa        | causa          
incendios       | id_vegetacion   | vegetacion     
municipios      | id_clave_ent    | estado         
operaciones     | id_clave_inc    | incendios      


## Huerfanos

In [ ]:
print(" INICIANDO AUDITORÍA DINÁMICA DE INTEGRIDAD...\n")
hay_problemas = False

for hija, fk, padre, pk in relaciones:
    query = f"""
        SELECT COUNT(*) 
        FROM main.{hija} h
        LEFT JOIN main.{padre} p ON h.{fk} = p.{pk}
        WHERE p.{pk} IS NULL AND h.{fk} IS NOT NULL
    """
    
    total_huerfanos = con.execute(query).fetchone()[0]
    
    if total_huerfanos > 0:
        hay_problemas = True
        print(f"TABLA: {hija:<12} | FK: {fk:<15} | HUÉRFANOS: {total_huerfanos}")
        
        # Obtenemos una muestra de los IDs que no existen en el padre
        muestras = con.execute(f"""
            SELECT DISTINCT h.{fk} 
            FROM main.{hija} h 
            LEFT JOIN main.{padre} p ON h.{fk} = p.{pk} 
            WHERE p.{pk} IS NULL LIMIT 3
        """).fetchall()
        print(f" IDs que no están en '{padre}': {[m[0] for m in muestras]}")
        print("-" * 60)
    else:
        print(f"TABLA: {hija:<12} | Relación con '{padre}' perfecta.")

if not hay_problemas:
    print("\nNo hay registros huérfanos.")
else:
    print("\nSe encontraron huérfanos. Necesitamos decidir si borrar o corregir.")

 INICIANDO AUDITORÍA DINÁMICA DE INTEGRIDAD...

TABLA: climatologia | Relación con 'diccionario' perfecta.
TABLA: climatologia | Relación con 'incendios' perfecta.
TABLA: danos        | Relación con 'incendios' perfecta.
TABLA: demografia   | Relación con 'municipios' perfecta.
TABLA: incendios    | Relación con 'municipios' perfecta.
TABLA: incendios    | Relación con 'causa' perfecta.
TABLA: incendios    | Relación con 'vegetacion' perfecta.
TABLA: municipios   | Relación con 'estado' perfecta.
TABLA: operaciones  | Relación con 'incendios' perfecta.

No hay registros huérfanos.


Nivel Crítico (climatologia): Aquí es donde más fallan los procesos ETL. Si un satélite mandó dos veces el mismo dato de temperatura para el mismo incendio, tus promedios se sesgan. Aquí sí es obligatorio limpiar.

Nivel Estructural (municipios, causa, vegetacion): Si estas tienen duplicados, tus JOINs van a crear un "efecto explosión" (una fila se convierte en dos al unir), duplicando artificialmente tus hectáreas quemadas.

Nivel de Hechos (incendios, danos): Si aquí hay duplicados, tu conteo de incendios anuales será mentira.

### Duplicados 

In [ ]:
tablas_a_revisar = {
    'climatologia': ['id_variable', 'id_clave_inc', 'fecha_de_observacion'],
    'danos': ['id_clave_inc'], # Aquí solo debería haber uno por incendio
    'demografia': ['id_cvegeo', 'anio', 'sexo'],
    'operaciones': ['id_clave_inc']
}
for tabla, columnas in tablas_a_revisar.items():
    cols_str = ", ".join(columnas)
    query = f"""
        SELECT {cols_str}, COUNT(*) as repeticiones
        FROM main.{tabla}
        GROUP BY {cols_str}
        HAVING repeticiones > 1
        LIMIT 5
    """
    res = con.execute(query).df()
    
    if not res.empty:
        print(f"TABLA: {tabla.upper()} -> ¡Se encontraron duplicados!")
        print(res)
    else:
        print(f"TABLA: {tabla.upper()} -> Sin duplicados.")

TABLA: CLIMATOLOGIA -> ¡Se encontraron duplicados!
   id_variable id_clave_inc fecha_de_observacion  repeticiones
0  PRECTOTCORR   15-15-0018           2015-01-23             2
1  PRECTOTCORR   15-15-0031           2015-01-29             2
2  PRECTOTCORR   15-15-0039           2015-01-29             2
3  PRECTOTCORR   15-15-0037           2015-01-31             2
4  PRECTOTCORR   15-15-0044           2015-01-30             2
TABLA: DANOS -> Sin duplicados.
TABLA: DEMOGRAFIA -> Sin duplicados.
TABLA: OPERACIONES -> Sin duplicados.


# Lógica Temporal (Por el momento no se limpian)

In [ ]:
import pandas as pd
import requests
import json
import os

# 1. Cargar IDs desde el diccionario local
JSON_PATH = '/home/saul/Metodologia/IncendiosForestales/01_ExtraccionDeDatos/DatosNASA/raw/diccionario_verificado.json'

if not os.path.exists(JSON_PATH):
    print(f"Error: No se encuentra el archivo en {JSON_PATH}")
else:
    with open(JSON_PATH, 'r', encoding='utf-8') as f:
        diccionario_local = json.load(f)
    ids_variables = [v['id'] for v in diccionario_local]

def extraer_metadata_vía_point(lista_ids):
    # Paquete de prueba con las primeras 20 variables
    bundle = ",".join(lista_ids[:20]) 
    
    # Coordenadas de referencia
    lat, lon = 19.4, -99.1
    test_url = (f"https://power.larc.nasa.gov/api/temporal/daily/point?"
                f"start=20230101&end=20230101&latitude={lat}&longitude={lon}"
                f"&community=ag&parameters={bundle}&format=json")
    
    print("Consultando definiciones y unidades oficiales a la NASA...")
    
    try:
        res = requests.get(test_url, timeout=30)
        if res.status_code == 200:
            data = res.json()
            
            # --- CORRECCIÓN: Acceso a metadatos en la raíz del JSON ---
            meta_dict = data.get('parameters', {})
            
            if not meta_dict:
                # Intento en ruta alternativa segun version de API
                meta_dict = data.get('header', {}).get('parameter', {})
            
            if not meta_dict:
                print("No se encontraron metadatos en la respuesta de la API.")
                return None
            
            # Crear DataFrame y transponer para que los IDs sean filas
            df_meta = pd.DataFrame.from_dict(meta_dict, orient='index')
            
            # Seleccionar columnas de interes para el analisis de limpieza
            columnas_finales = ['long_name', 'units']
            return df_meta[columnas_finales] if all(c in df_meta.columns for c in columnas_finales) else df_meta
            
        else:
            print(f"Error de API: {res.status_code}")
            return None
    except Exception as e:
        print(f"Fallo en la conexion: {e}")
        return None

# 2. Ejecutar y mostrar resultados en Jupyter
df_resultados = extraer_metadata_vía_point(ids_variables)

if df_resultados is not None:
    print("\nMetadatos recuperados (Unidades y Nombres):")
    from IPython.display import display
    display(df_resultados)

Consultando definiciones y unidades oficiales a la NASA...

Metadatos recuperados (Unidades y Nombres):


,units,longname
T2M,C,Temperature at 2 Meters
ALLSKY_SFC_SW_DWN,MJ/m^2/day,All Sky Surface Shortwave Downward Irradiance
RH2M,%,Relative Humidity at 2 Meters
WS2M,m/s,Wind Speed at 2 Meters
T2MDEW,C,Dew/Frost Point at 2 Meters
T2M_MAX,C,Temperature at 2 Meters Maximum
T2M_MIN,C,Temperature at 2 Meters Minimum
PS,kPa,Surface Pressure
T2MWET,C,Wet Bulb Temperature at 2 Meters
WS50M,m/s,Wind Speed at 50 Meters


In [ ]:
import duckdb
import pandas as pd

# 1. Conexión con la ruta y modo especificado
con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

# 2. Definición de la consulta de inspección
query = """
SELECT 
    d.id_variable, 
    d.nombre_completo, 
    d.unidad_de_medida,
    COUNT(*) as total_registros,
    ROUND(AVG(c.resultado_numerico), 2) as promedio,
    MIN(c.resultado_numerico) as valor_min,
    MAX(c.resultado_numerico) as valor_max
FROM main.climatologia c
JOIN main.diccionario d ON c.id_variable = d.id_variable
GROUP BY ALL
ORDER BY total_registros DESC;
"""

try:
    # 3. Ejecución y visualización
    df_inspeccion = con.execute(query).df()
    
    # Ajuste para que Jupyter muestre todas las filas si es necesario
    pd.set_option('display.max_rows', 200)
    
    display(df_inspeccion)

finally:
    # 4. Cierre de conexión
    con.close()

,id_variable,nombre_completo,unidad_de_medida,total_registros,promedio,valor_min,valor_max
0,PRECTOTCORR,precipitation corrected,mm/day,197130,0.48,0.000000,46.049999
1,WS10M_MIN,wind speed at 10 meters minimum,m/s,98570,0.79,0.010000,9.230000
2,CLRSKY_KT,clear sky insolation clearness index,dimensionless,98570,0.74,0.490000,0.810000
3,IMERG_PRECTOT,total precipitation,mm/day,98570,-999.00,-999.000000,-999.000000
4,QV2M,specific humidity at 2 meters,g/kg,98570,6.51,2.300000,20.860001
5,CLRSKY_SFC_SW_DWN,clear sky surface shortwave downward irradiance,mj/m^2/day,98570,26.27,17.270000,31.420000
6,WS50M_MIN,wind speed at 50 meters minimum,m/s,98570,1.20,0.000000,12.890000
7,CLRSKY_SFC_PAR_TOT,clear sky surface total par,mj/m^2/day,98570,11.32,7.310000,13.610000
8,TS_MIN,earth skin temperature minimum,c,98570,7.55,-6.270000,30.270000
9,AOD_55,aerosol optical depth 55,dimensionless,98570,0.41,0.020000,1.460000


In [ ]:
import duckdb
import pandas as pd

# Conexión en modo lectura para seguridad total
con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

# 1. Consulta para detectar el "Ruido NASA" (-999) en las 146 variables
query_suciedad = """
WITH Estadisticas AS (
    SELECT 
        id_variable,
        COUNT(*) as total_registros,
        SUM(CASE WHEN resultado_numerico = -999.0 THEN 1 ELSE 0 END) as conteo_999,
        MIN(resultado_numerico) FILTER (WHERE resultado_numerico != -999.0) as min_real,
        MAX(resultado_numerico) FILTER (WHERE resultado_numerico != -999.0) as max_real
    FROM main.climatologia
    GROUP BY id_variable
)
SELECT 
    e.id_variable,
    d.nombre_completo,
    e.total_registros,
    e.conteo_999,
    ROUND((e.conteo_999 * 100.0) / e.total_registros, 2) as porcentaje_error_nasa,
    e.min_real,
    e.max_real,
    d.unidad_de_medida
FROM Estadisticas e
JOIN main.diccionario d ON e.id_variable = d.id_variable
ORDER BY porcentaje_error_nasa DESC;
"""

try:
    df_diagnostico = con.execute(query_suciedad).df()
    
    # Filtrar las "Variables Críticas" (más del 20% de error)
    variables_criticas = df_diagnostico[df_diagnostico['porcentaje_error_nasa'] > 20]
    
    print(f"📊 Diagnóstico de 146 variables completado.")
    print(f"⚠️ Se detectaron {len(variables_criticas)} variables con alto índice de error (>20%).")
    
    display(df_diagnostico)
finally:
    con.close()

📊 Diagnóstico de 146 variables completado.
⚠️ Se detectaron 23 variables con alto índice de error (>20%).


,id_variable,nombre_completo,total_registros,conteo_999,porcentaje_error_nasa,min_real,max_real,unidad_de_medida
0,GWM_HEIGHT,gwm height,98570,98570.0,100.00,NaN,NaN,m
1,RZMC_PRCNTL,root zone soil moisture (percentile),98550,98550.0,100.00,NaN,NaN,%
2,SZA,solar zenith angle,98570,98570.0,100.00,NaN,NaN,degrees
3,PRMC,profile soil moisture,98550,98550.0,100.00,NaN,NaN,m3 m-3
4,CLOUD_BT,cloud bottom temperature,98560,98560.0,100.00,NaN,NaN,c
5,CLOUD_BT_MAX,maximum cloud bottom temperature,98560,98560.0,100.00,NaN,NaN,c
6,CLOUD_BT_MIN,minimum cloud bottom temperature,98560,98560.0,100.00,NaN,NaN,c
7,IMERG_PRECTOT,total precipitation,98570,98570.0,100.00,NaN,NaN,mm/day
8,CLOUD_TT_MAX,maximum cloud top temperature,98560,98560.0,100.00,NaN,NaN,c
9,CLOUD_TT_MIN,minimum cloud top temperature,98560,98560.0,100.00,NaN,NaN,c


In [ ]:
import duckdb

# Conexión
con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

# Consultamos todas las unidades únicas
query_unidades = """
SELECT 
    unidad_de_medida, 
    COUNT(*) as cantidad_de_variables,
    GROUP_CONCAT(id_variable) as ejemplos
FROM main.diccionario
GROUP BY unidad_de_medida
ORDER BY cantidad_de_variables DESC;
"""

df_unidades = con.execute(query_unidades).df()
con.close()

print("📋 Unidades detectadas en el diccionario:")
display(df_unidades)

📋 Unidades detectadas en el diccionario:


,unidad_de_medida,cantidad_de_variables,ejemplos
0,mj/m^2/day,35,"ALLSKY_SFC_SW_DWN,ALLSKY_SFC_LW_DWN,ALLSKY_SFC_SW_DIFF,ALLSKY_SFC_SW_DNI,TOA_SW_DWN,ALLSKY_SFC_PAR_TOT,CLRSKY_SFC_SW_DWN,ALLSKY_SFC_UVA,ALLSKY_SFC_UVB,CLRSKY_SFC_PAR_TOT,MIDDAY_INSOL,TOA_SW_DNI,ALLSKY_SFC_LW_UP,ALLSKY_SFC_PAR_DIFF,ALLSKY_SFC_PAR_DIRH,ALLSKY_SFC_SW_DIRH,ALLSKY_SFC_SW_UP,CLRSKY_SFC_LW_DWN,CLRSKY_SFC_LW_UP,CLRSKY_SFC_PAR_DIFF,CLRSKY_SFC_PAR_DIRH,CLRSKY_SFC_SW_DIFF,CLRSKY_SFC_SW_DIRH,CLRSKY_SFC_SW_DNI,CLRSKY_SFC_SW_UP,EVPTRNS,LWLAND,ORIGINAL_ALLSKY_SFC_LW_DWN,ORIGINAL_ALLSKY_SFC_SW_DIFF,ORIGINAL_ALLSKY_SFC_SW_DIRH,ORIGINAL_ALLSKY_SFC_SW_DWN,ORIGINAL_CLRSKY_SFC_LW_DWN,ORIGINAL_CLRSKY_SFC_SW_DWN,PSH,SWLAND"
1,c,29,"T2M,T2MDEW,T2M_MAX,T2M_MIN,T2MWET,TS,T2M_RANGE,TS_MAX,TS_MIN,CLOUD_BT,CLOUD_BT_MAX,CLOUD_BT_MIN,CLOUD_TT,CLOUD_TT_MAX,CLOUD_TT_MIN,T10M,T10M_MAX,T10M_MIN,T10M_RANGE,TROPT,TSOIL1,TSOIL2,TSOIL3,TSOIL4,TSOIL5,TSOIL6,TSURF,TS_ADJ,TS_RANGE"
2,m/s,18,"WS2M,WS50M,WS10M,WS10M_MAX,WS10M_MIN,WS10M_RANGE,WS50M_MAX,WS50M_MIN,WS50M_RANGE,WS2M_MAX,WS2M_MIN,WS2M_RANGE,U10M,V10M,U2M,U50M,V2M,V50M"
3,dimensionless,12,"ALLSKY_SRF_ALB,ALLSKY_KT,CLRSKY_KT,AOD_55,AIRMASS,ALLSKY_NKT,AOD_55_ADJ,AOD_84,CLOUD_OD,CLRSKY_NKT,CLRSKY_SRF_ALB,SRF_ALB_ADJ"
4,%,7,"RH2M,CLOUD_AMT,CLOUD_AMT_DAY,CLOUD_AMT_NIGHT,IMERG_PRECLIQUID_PROB,PRMC_PRCNTL,RZMC_PRCNTL"
5,1,5,"GWETPROF,GWETROOT,GWETTOP,FRSEAICE,FRSNO"
6,mm/day,5,"PRECTOTCORR,IMERG_PRECTOT,EVLAND,PRECSNO,PRECSNOLAND"
7,m,4,"GWM_HEIGHT,DISPH,GWM_HEIGHT_ANOMALY,Z0M"
8,degrees,4,"WD50M,WD2M,WD10M,SZA"
9,kpa,4,"PS,PBLTOP,SLP,TROPPB"


In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

query_auditoria_total = """
WITH Clasificacion AS (
    SELECT 
        c.id_variable,
        d.unidad_de_medida,
        c.resultado_numerico,
        CASE 
            WHEN c.resultado_numerico = -999.0 THEN 'Ruido NASA (-999)'
            
            -- Grupo: No pueden ser negativos
            WHEN LOWER(d.unidad_de_medida) IN ('m/s', 'mj/m^2/day', 'mm/day', 'kpa', 'g/kg', 'm3 m-3', 'cm', 'days', 'kg m-2', 'count', 'w m-2 x 40', 'kg m-3', 'dobsons', 'degree-day-c', 'm') 
                 AND c.resultado_numerico < 0 THEN 'Valor Negativo Imposible'
            
            -- Grupo: Porcentajes
            WHEN d.unidad_de_medida = '%' 
                 AND (c.resultado_numerico < 0 OR c.resultado_numerico > 100) THEN 'Fuera de Rango [0-100%]'
            
            -- Grupo: Índices Normalizados (NDVI, etc.)
            WHEN d.unidad_de_medida = 'adimensional (-1 a 1)' 
                 AND (c.resultado_numerico < -1 OR c.resultado_numerico > 1) THEN 'Indice fuera de rango [-1, 1]'
            
            -- Grupo: Ángulos
            WHEN d.unidad_de_medida = 'degrees' 
                 AND (c.resultado_numerico < 0 OR c.resultado_numerico > 360) THEN 'Angulo Invalido [0-360]'
            
            -- Grupo: Fracciones (Unidad '1' como GWETROOT o dimensionless)
            WHEN (d.unidad_de_medida = '1' OR d.unidad_de_medida = 'dimensionless')
                 AND (c.resultado_numerico < 0 OR c.resultado_numerico > 100) THEN 'Fraccion/Ratio fuera de rango'

            ELSE 'Dato Valido'
        END AS diagnostico
    FROM main.climatologia c
    JOIN main.diccionario d ON c.id_variable = d.id_variable
)
SELECT 
    id_variable,
    unidad_de_medida,
    diagnostico,
    COUNT(*) AS total_hallazgos,
    ROUND(MIN(resultado_numerico), 4) AS min_detectado,
    ROUND(MAX(resultado_numerico), 4) AS max_detectado
FROM Clasificacion
WHERE diagnostico != 'Dato Valido'
GROUP BY ALL
ORDER BY total_hallazgos DESC;
"""

try:
    df_reporte = con.execute(query_auditoria_total).df()
    print("📋 REPORTE DE INSPECCIÓN FÍSICA (146 VARIABLES)")
    if df_reporte.empty:
        print("✅ No se detectaron anomalías físicas en el dataset.")
    else:
        display(df_reporte)
finally:
    con.close()

📋 REPORTE DE INSPECCIÓN FÍSICA (146 VARIABLES)


,id_variable,unidad_de_medida,diagnostico,total_hallazgos,min_detectado,max_detectado
0,IMERG_PRECTOT,mm/day,Ruido NASA (-999),98570,-999.00,-999.00
1,GWM_HEIGHT,m,Ruido NASA (-999),98570,-999.00,-999.00
2,SZA,degrees,Ruido NASA (-999),98570,-999.00,-999.00
3,CLOUD_BT_MIN,c,Ruido NASA (-999),98560,-999.00,-999.00
4,CLOUD_BT_MAX,c,Ruido NASA (-999),98560,-999.00,-999.00
5,CLOUD_TT_MAX,c,Ruido NASA (-999),98560,-999.00,-999.00
6,CLOUD_BT,c,Ruido NASA (-999),98560,-999.00,-999.00
7,CLOUD_TT_MIN,c,Ruido NASA (-999),98560,-999.00,-999.00
8,CLOUD_TT,c,Ruido NASA (-999),98560,-999.00,-999.00
9,SWLAND,mj/m^2/day,Ruido NASA (-999),98550,-999.00,-999.00


In [ ]:
import duckdb
import pandas as pd

# 1. Conexión segura (Simulación total, nada se escribe en disco)
con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

# 2. Consulta para calcular la "Salud" de las 146 variables
query_simulacion = """
SELECT 
    id_variable,
    COUNT(*) AS total_registros,
    SUM(CASE WHEN resultado_numerico = -999.0 THEN 1 ELSE 0 END) AS nulos_simulados
FROM main.climatologia
GROUP BY id_variable
"""

try:
    # 3. Procesamiento en Pandas para aplicar tu lógica
    df_salud = con.execute(query_simulacion).df()
    
    # Calcular porcentaje de nulos
    df_salud['porcentaje_nulos'] = (df_salud['nulos_simulados'] / df_salud['total_registros']) * 100
    
    # Aplicar tus reglas de decisión
    def definir_estrategia(row):
        if row['porcentaje_nulos'] > 50:
            return "❌ DESCARTAR (Variable muerta)"
        elif row['porcentaje_nulos'] < 5:
            return "🩹 INTERPOLAR (Rescatable)"
        else:
            return "🔍 REVISAR (Estrategia mixta)"

    df_salud['estrategia_sugerida'] = df_salud.apply(definir_estrategia, axis=1)
    
    # 4. Resumen ejecutivo para tu reporte
    resumen = df_salud['estrategia_sugerida'].value_counts()
    print("--- RESUMEN DE ESTRATEGIA DE DATOS ---")
    print(resumen)
    print("-" * 40)
    
    # Mostrar tabla detallada ordenada por las más dañadas
    display(df_salud.sort_values(by='porcentaje_nulos', ascending=False))

finally:
    con.close()

--- RESUMEN DE ESTRATEGIA DE DATOS ---
estrategia_sugerida
🩹 INTERPOLAR (Rescatable)        120
❌ DESCARTAR (Variable muerta)     23
🔍 REVISAR (Estrategia mixta)       2
Name: count, dtype: int64
----------------------------------------


,id_variable,total_registros,nulos_simulados,porcentaje_nulos,estrategia_sugerida
14,CLOUD_TT_MIN,98560,98560.0,100.000000,❌ DESCARTAR (Variable muerta)
88,IMERG_PRECLIQUID_PROB,98550,98550.0,100.000000,❌ DESCARTAR (Variable muerta)
86,CLOUD_TT,98560,98560.0,100.000000,❌ DESCARTAR (Variable muerta)
101,IMERG_PRECTOT_COUNT,98550,98550.0,100.000000,❌ DESCARTAR (Variable muerta)
102,ORIGINAL_CLRSKY_SFC_LW_DWN,98550,98550.0,100.000000,❌ DESCARTAR (Variable muerta)
117,ORIGINAL_ALLSKY_SFC_LW_DWN,98550,98550.0,100.000000,❌ DESCARTAR (Variable muerta)
118,ORIGINAL_ALLSKY_SFC_SW_DWN,98550,98550.0,100.000000,❌ DESCARTAR (Variable muerta)
119,ORIGINAL_CLRSKY_SFC_SW_DWN,98550,98550.0,100.000000,❌ DESCARTAR (Variable muerta)
77,GWM_HEIGHT,98570,98570.0,100.000000,❌ DESCARTAR (Variable muerta)
55,CLOUD_TT_MAX,98560,98560.0,100.000000,❌ DESCARTAR (Variable muerta)


In [ ]:
import duckdb

con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

query_duplicados = """
SELECT 
    id_variable, 
    id_clave_inc, 
    fecha_de_observacion, 
    COUNT(*) as repeticiones
FROM main.climatologia
GROUP BY id_variable, id_clave_inc, fecha_de_observacion
HAVING COUNT(*) > 1
ORDER BY repeticiones DESC
LIMIT 20;
"""

df_dupes = con.execute(query_duplicados).df()
con.close()

if df_dupes.empty:
    print("✅ ¡Limpio! No hay registros duplicados por combinación de fecha/variable.")
else:
    print("⚠️ Se detectaron registros duplicados. Aquí los más repetidos:")
    display(df_dupes)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

⚠️ Se detectaron registros duplicados. Aquí los más repetidos:


,id_variable,id_clave_inc,fecha_de_observacion,repeticiones
0,PRECTOTCORR,22-15-0829,2022-05-01,3
1,PRECTOTCORR,20-15-0203,2020-03-12,3
2,PRECTOTCORR,22-15-0829,2022-05-03,3
3,PRECTOTCORR,20-15-0203,2020-03-16,3
4,PRECTOTCORR,22-15-0829,2022-04-28,3
5,PRECTOTCORR,22-15-0829,2022-04-25,3
6,PRECTOTCORR,20-15-0203,2020-03-17,3
7,PRECTOTCORR,20-15-0203,2020-03-15,3
8,PRECTOTCORR,22-15-0829,2022-04-26,3
9,PRECTOTCORR,22-15-0829,2022-04-27,3


# Valores Numericos 

## Climatologia

In [ ]:
def limpieza_sistemica_total(con):
    print("🚀 Iniciando limpieza sistémica de la base de datos...")

    # 1. Duplicados en tablas 1:1 (Incendios, Daños, Operaciones)
    tablas_clave = ['incendios', 'danos', 'operaciones']
    for tabla in tablas_clave:
        con.execute(f"""
            DELETE FROM main.{tabla} 
            WHERE rowid NOT IN (
                SELECT min(rowid) FROM main.{tabla} GROUP BY id_clave_inc
            )
        """)
    
    # 2. Duplicados en Climatología (EAV: id_variable + incendio + fecha)
    print("   - Limpiando duplicados en Climatología (17M registros)...")
    con.execute("""
        DELETE FROM main.climatologia 
        WHERE rowid NOT IN (
            SELECT min(rowid) 
            FROM main.climatologia 
            GROUP BY id_variable, id_clave_inc, fecha_de_observacion
        )
    """)

    # 3. Neutralizar NASA
    con.execute("UPDATE main.climatologia SET resultado_numerico = NULL WHERE resultado_numerico = -999.0")

    # 4. Borrar Variables 100% Muertas
    print("   - Identificando y eliminando variables sin datos...")
    con.execute("""
        DELETE FROM main.diccionario 
        WHERE id_variable IN (
            SELECT id_variable FROM main.climatologia 
            GROUP BY id_variable HAVING COUNT(resultado_numerico) = 0
        )
                
SELECT 
    id_variable,
    COUNT(*) AS total_registros,
    SUM(CASE WHEN resultado_numerico = -999.0 THEN 1 ELSE 0 END) AS nulos_simulados
FROM main.climatologia
GROUP BY id_variable
"""
                

    """)
    con.execute("""
        DELETE FROM main.climatologia 
        WHERE id_variable NOT IN (SELECT id_variable FROM main.diccionario)
    """)

    #Falto el filtro en el caso de climatologia es decir la interpolacion lineal para recartar aquellos que sean rescatables 
    #Revisar el ajuste de la unificacion de municipios

    # 5. Integridad Referencial Final (Huérfanos)
    print("   - Eliminando registros huérfanos...")
    con.execute("DELETE FROM main.climatologia WHERE id_clave_inc NOT IN (SELECT id_clave_inc FROM main.incendios)")


import duckdb
import pandas as pd

# 1. Conexión segura (Simulación total, nada se escribe en disco)
con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

# 2. Consulta para calcular la "Salud" de las 146 variables
query_simulacion = """
SELECT 
    id_variable,
    COUNT(*) AS total_registros,
    SUM(CASE WHEN resultado_numerico = -999.0 THEN 1 ELSE 0 END) AS nulos_simulados
FROM main.climatologia
GROUP BY id_variable
"""
()

import duckdb

def ejecutar_limpieza_final(path_db):
    con = duckdb.connect(path_db)
    print("🚀 Iniciando Limpieza Sistémica (Nivel IPN)...")

    # 1. CORRECCIÓN GEOGRÁFICA (Municipios e Incendios)
    # Volteamos las claves invertidas (Ej: 10215 -> 15102)
    print("   [1/5] Corrigiendo claves geográficas invertidas...")
    con.execute("""
        UPDATE main.incendios 
        SET id_cvegeo = SUBSTR(id_cvegeo, 4, 2) || LPAD(SUBSTR(id_cvegeo, 1, 3), 3, '0')
        WHERE id_cvegeo LIKE '%15' AND id_cvegeo NOT LIKE '15%' AND LENGTH(id_cvegeo) = 5;
        
        UPDATE main.demografia 
        SET id_cvegeo = SUBSTR(id_cvegeo, 4, 2) || LPAD(SUBSTR(id_cvegeo, 1, 3), 3, '0')
        WHERE id_cvegeo LIKE '%15' AND id_cvegeo NOT LIKE '15%' AND LENGTH(id_cvegeo) = 5;
    """)

    # 2. DEDUPLICACIÓN DE CATÁLOGOS (Causa y Vegetación)
    print("   [2/5] Fusionando duplicados lógicos en catálogos...")
    for tabla, cols in {'causa': ['causa', 'causa_especifica'], 
                        'vegetacion': ['regimen_del_fuego', 'tipo_de_vegetacion']}.items():
        id_col = f"id_{tabla}"
        cols_sql = ", ".join(cols)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE mapeo AS
            SELECT {id_col} as id_malo, 
                   FIRST_VALUE({id_col}) OVER (PARTITION BY {cols_sql} ORDER BY {id_col}) as id_bueno
            FROM main.{tabla};
            
            UPDATE main.incendios SET {id_col} = m.id_bueno
            FROM mapeo m WHERE main.incendios.{id_col} = m.id_malo AND m.id_malo != m.id_bueno;
            
            DELETE FROM main.{tabla} WHERE {id_col} NOT IN (SELECT id_bueno FROM mapeo);
        """)

    # 3. LIMPIEZA DE TABLAS HIJAS (Diferencia de 1:1)
    print("   [3/5] Eliminando duplicados en Daños y Operaciones...")
    for t in ['incendios', 'danos', 'operaciones']:
        con.execute(f"DELETE FROM main.{t} WHERE rowid NOT IN (SELECT min(rowid) FROM main.{t} GROUP BY id_clave_inc)")

    # 4. NEUTRALIZACIÓN DE CLIMATOLOGÍA (Los 17 Millones)
    print("   [4/5] Procesando Climatología (Neutralizando NASA y Duplicados)...")
    con.execute("""
        -- Quitar duplicados por variable/incendio/fecha
        DELETE FROM main.climatologia 
        WHERE rowid NOT IN (SELECT min(rowid) FROM main.climatologia GROUP BY id_variable, id_clave_inc, fecha_de_observacion);
        
        -- Convertir NASA Noise a NULL
        UPDATE main.climatologia SET resultado_numerico = NULL WHERE resultado_numerico = -999.0;
    """)

    # 5. ELIMINACIÓN DE VARIABLES MUERTAS
    print("   [5/5] Eliminando variables del diccionario con 100% nulos...")
    con.execute("""
        DELETE FROM main.diccionario WHERE id_variable IN (
            SELECT id_variable FROM main.climatologia GROUP BY id_variable HAVING COUNT(resultado_numerico) = 0
        );
        DELETE FROM main.climatologia WHERE id_variable NOT IN (SELECT id_variable FROM main.diccionario);
    """)

    con.close()
    print("✅ ¡Base de Datos purificada y lista para el modelo!")

# Ejecutar
ejecutar_limpieza_final('../data/BaseDeDatos_Working.db')


In [5]:
import duckdb
import pandas as pd

# Conexión en modo lectura para inspección
con = duckdb.connect('../data/BaseDeDatos_Working.db', read_only=True)

# Definimos explícitamente el nombre de la tabla, su llave primaria real y sus columnas lógicas
catalogos_a_revisar = {
    'causa': {
        'pk': 'id_causa', 
        'cols': ['causa', 'causa_especifica']
    },
    'vegetacion': {
        'pk': 'id_vegetacion', 
        'cols': ['regimen_del_fuego', 'tipo_de_vegetacion']
    },
    'municipios': {
        'pk': 'id_cvegeo', 
        'cols': ['id_clave_ent', 'nombre_municipio']
    }
}

print("🕵️ Inspeccionando duplicados lógicos en catálogos...")

for tabla, config in catalogos_a_revisar.items():
    pk = config['pk']
    cols_str = ", ".join(config['cols'])
    
    # La consulta ahora usa la llave primaria correcta (id_causa, id_vegetacion o id_cvegeo)
    query = f"""
        SELECT {cols_str}, COUNT(*) as repeticiones, GROUP_CONCAT({pk}) as ids_afectados
        FROM main.{tabla}
        GROUP BY {cols_str}
        HAVING COUNT(*) > 1
    """
    
    try:
        res = con.execute(query).df()
        
        if not res.empty:
            print(f"\n⚠️ TABLA {tabla.upper()}: ¡Se encontraron duplicados lógicos!")
            display(res)
        else:
            print(f"✅ TABLA {tabla.upper()}: Limpia.")
            
    except Exception as e:
        print(f"❌ Error al procesar {tabla}: {e}")

con.close()

🕵️ Inspeccionando duplicados lógicos en catálogos...

⚠️ TABLA CAUSA: ¡Se encontraron duplicados lógicos!


,causa,causa_especifica,repeticiones,ids_afectados
0,intencional,vandalismo,3,"43,18,1"
1,otras actividades productivas,mineria (extraccion de materiales),2,"38,13"
2,fogatas,fogatas,3,"47,15,2"
3,otras causas,desconocidas,3,"6,53,55"
4,cazadores,ninguna / no aplica,2,"41,32"
5,fogatas,otras,2,"56,33"



⚠️ TABLA VEGETACION: ¡Se encontraron duplicados lógicos!


,regimen_del_fuego,tipo_de_vegetacion,repeticiones,ids_afectados
0,otros,matorral crasicaule,2,"19,8"



⚠️ TABLA MUNICIPIOS: ¡Se encontraron duplicados lógicos!


,id_clave_ent,nombre_municipio,repeticiones,ids_afectados
0,15,timilpan,2,"10215,15102"
1,15,zumpahuacan,2,"11915,15119"
2,15,zinacantepec,2,"11815,15118"
3,15,tonatico,2,"10715,15107"
4,15,tultitlan,2,"10915,15109"
5,15,toluca,2,"15106,10615"
6,15,tlalnepantla de baz,2,"15104,10415"
7,15,san jose del rincon,2,"15124,12415"
8,15,villa de allende,2,"11115,15111"
9,15,tlalmanalco,2,"15103,10315"
